# 03 · Monte Carlo vs TD
Guide: `01-beginner/04`, `03-expert/02`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
np.set_printoptions(precision=4, suppress=True)

In [ ]:
from rlguide import mc_td_prediction as mt
ep = [('A', 0), ('B', 2), ('A', -1), ('C', 3)]
print('returns', mt.returns_backward(ep))
print('first-visit', mt.mc_prediction([ep])[0])
print('every-visit', mt.mc_prediction([ep], first_visit=False)[0])

## Batch TD vs batch MC (Sutton & Barto Example 6.4)
> **Predict before you run:** what does each method say V(A) is?

In [ ]:
data = mt.EXAMPLE_6_4
print('batch MC:', mt.mc_prediction(data)[0])
print('batch TD:', {k: round(v, 4) for k, v in mt.batch_td0(data).items()})

## Random walk: RMS error vs episodes for different α

In [ ]:
def random_walk_episode(rng, n=5):
    s, ep = n // 2, []
    while True:
        s2 = s + (1 if rng.random() < 0.5 else -1)
        r = 1.0 if s2 == n else 0.0
        ep.append((s, r))
        if s2 in (-1, n):
            return ep
        s = s2

true_v = np.arange(1, 6) / 6

@interact(alpha=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01))
def compare(alpha):
    rng = np.random.default_rng(0)
    V_td, V_mc, e_td, e_mc = np.full(5, .5), np.full(5, .5), [], []
    for _ in range(100):
        ep = random_walk_episode(rng)
        for t, (s, r) in enumerate(ep):
            nxt = V_td[ep[t + 1][0]] if t + 1 < len(ep) else 0.0
            V_td[s] += alpha * (r + nxt - V_td[s])
        G = mt.returns_backward(ep)
        for (s, _), g in zip(ep, G):
            V_mc[s] += alpha * (g - V_mc[s])
        e_td.append(np.sqrt(((V_td - true_v)**2).mean()))
        e_mc.append(np.sqrt(((V_mc - true_v)**2).mean()))
    plt.plot(e_td, label='TD(0)'); plt.plot(e_mc, label='constant-alpha MC')
    plt.ylabel('RMS error'); plt.xlabel('episode'); plt.legend(); plt.show()